In [2]:
from pathlib import Path
from sqlalchemy import create_engine

# Build the path and engine
db_path = Path.cwd().parent / 'data' / 'database' / 'retail.db'
engine = create_engine(f"sqlite:///{db_path}")

# Load the extension and connect using the engine object directly
%load_ext sql
%sql engine

# Confirm connection and check what tables exist
%sql SELECT name FROM sqlite_master WHERE type='table'

Running query in 'sqlite:////Users/sofiaconcepcion/Documents/GitHub/Australian-Retail-Trade-Performance/data/database/retail.db'

name
retail_turnover


### Query 1: YoY Growth by Category

💡 Which retail categorie grew faster or slower than last year, and by how much?

In [15]:
%%sql
-- Create helping column annual which aggregates sales per category per year


WITH annual AS (
    SELECT
        year,
        category,
        SUM(turnover_m) AS turnover_m
    FROM retail_turnover
    WHERE category != 'Total (Industry)' -- exclude the aggregate row
        AND year < 2025                   
    GROUP BY year, category
)
SELECT
    a.year,
    a.category,
    CAST(ROUND(a.turnover_m, 1) AS REAL) AS turnover_m,
    CAST(ROUND(b.turnover_m, 1) AS REAL) AS prior_year_m,
    CAST(ROUND((a.turnover_m - b.turnover_m) / b.turnover_m * 100, 1) AS REAL) AS yoy_growth_pct
FROM annual a
JOIN annual b 
    ON a.category = b.category
    AND a.year = b.year + 1
ORDER BY a.year DESC, yoy_growth_pct desc;

Running query in 'sqlite:////Users/sofiaconcepcion/Documents/GitHub/Australian-Retail-Trade-Performance/data/database/retail.db'

year,category,turnover_m,prior_year_m,yoy_growth_pct
2024,Other retailing,68326.2,64758.1,5.5
2024,Food retailing,173604.2,168448.9,3.1
2024,"Cafes, restaurants and takeaway food services",65411.7,63985.5,2.2
2024,"Clothing, footwear and personal accessory retailing",36146.3,35610.7,1.5
2024,Household goods retailing,70416.0,69536.0,1.3
2024,Department stores,22853.9,22687.4,0.7
2023,"Cafes, restaurants and takeaway food services",63985.5,58154.2,10.0
2023,Food retailing,168448.9,160936.8,4.7
2023,Department stores,22687.4,21950.4,3.4
2023,"Clothing, footwear and personal accessory retailing",35610.7,34756.8,2.5


### Query 2: Current Underperformers vs National Benchmark

💡 Which categories are growing slower than the national total right now, over the past 12 months?


In [27]:
%%sql

WITH last_date AS (
    -- find the most recent month actually in the database
    SELECT MAX(date) AS max_date FROM retail_turnover
),
recent AS (
    SELECT category, SUM(turnover_m) AS turnover_12m
    FROM retail_turnover, last_date
    WHERE date > date(max_date, '-12 months')
      AND date <= max_date
    GROUP BY category
),
prior AS (
    SELECT category, SUM(turnover_m) AS turnover_prior_12m
    FROM retail_turnover, last_date
    WHERE date > date(max_date, '-24 months')
      AND date <= date(max_date, '-12 months')
    GROUP BY category
)
SELECT
    r.category,
    CAST(ROUND(r.turnover_12m, 1) AS REAL)                         AS last_12m_m,
    CAST(ROUND((r.turnover_12m - p.turnover_prior_12m)
         / p.turnover_prior_12m * 100, 1) AS REAL)                 AS yoy_growth_pct
FROM recent r
JOIN prior p ON r.category = p.category
WHERE r.category != 'Total (Industry)'
ORDER BY yoy_growth_pct DESC

Running query in 'sqlite:////Users/sofiaconcepcion/Documents/GitHub/Australian-Retail-Trade-Performance/data/database/retail.db'

category,last_12m_m,yoy_growth_pct
Other retailing,75397.6,14.5
Food retailing,189895.4,11.4
Household goods retailing,77225.8,11.4
"Clothing, footwear and personal accessory retailing",39588.9,10.9
"Cafes, restaurants and takeaway food services",71388.5,10.4
Department stores,25001.0,10.2


### Query 3: Seasonal Pattern by Category

💡 What is the average turnover for each category in each month of the year? When does each category
naturally peak and trough?


In [28]:
%%sql
SELECT
    category,
    month_name,
    month,
    CAST(ROUND(AVG(turnover_m), 1) AS REAL) AS avg_monthly_turnover_m
FROM retail_turnover
WHERE category != 'Total (Industry)'
    AND year >= 2015
    AND year < 2025 -- exclude incomplete current year
GROUP BY category, month, month_name
ORDER BY category, month;

Running query in 'sqlite:////Users/sofiaconcepcion/Documents/GitHub/Australian-Retail-Trade-Performance/data/database/retail.db'

category,month_name,month,avg_monthly_turnover_m
"Cafes, restaurants and takeaway food services",Jan,1,3996.1
"Cafes, restaurants and takeaway food services",Feb,2,3702.8
"Cafes, restaurants and takeaway food services",Mar,3,4039.1
"Cafes, restaurants and takeaway food services",Apr,4,3879.2
"Cafes, restaurants and takeaway food services",May,5,3984.7
"Cafes, restaurants and takeaway food services",Jun,6,3917.2
"Cafes, restaurants and takeaway food services",Jul,7,4121.2
"Cafes, restaurants and takeaway food services",Aug,8,4121.4
"Cafes, restaurants and takeaway food services",Sep,9,4158.4
"Cafes, restaurants and takeaway food services",Oct,10,4310.3


### Query 3: Seasonal Pattern by Category

💡 What percentage of total retail turnover does each category represent, and how has that share shifted year
by year?


In [29]:
%%sql 

-- get annual totals for every category including Total (Industry)
WITH annual AS (
    SELECT 
        year, 
        category, 
        SUM(turnover_m) AS turnover_m
    FROM retail_turnover
    WHERE year < 2025 -- exclude incomplete current year
    GROUP BY year, category
)
SELECT
    a.year,
    a.category,
    --divide this category's total by the national total for the same year
    CAST(ROUND(a.turnover_m / t.turnover_m * 100, 1) AS REAL) AS share_of_total_pct
FROM annual a
-- self-join — 'a' is the individual category, 't' is Total (Industry)
    JOIN annual t
    ON a.year = t.year
    AND t.category = 'Total (Industry)'
WHERE a.category != 'Total (Industry)'
ORDER BY a.year DESC, share_of_total_pct DESC;

Running query in 'sqlite:////Users/sofiaconcepcion/Documents/GitHub/Australian-Retail-Trade-Performance/data/database/retail.db'

year,category,share_of_total_pct
2024,Food retailing,39.7
2024,Household goods retailing,16.1
2024,Other retailing,15.6
2024,"Cafes, restaurants and takeaway food services",15.0
2024,"Clothing, footwear and personal accessory retailing",8.3
2024,Department stores,5.2
2023,Food retailing,39.6
2023,Household goods retailing,16.4
2023,Other retailing,15.2
2023,"Cafes, restaurants and takeaway food services",15.1


In [8]:
%%sql

SELECT
    year,
    category,
    turnover_m,
    month
FROM retail_turnover
WHERE (month = 1 OR month = 12)
    AND year = 2024
    

Running query in 'sqlite:////Users/sofiaconcepcion/Documents/GitHub/Australian-Retail-Trade-Performance/data/database/retail.db'

year,category,turnover_m,month
2024,"Cafes, restaurants and takeaway food services",5219.1,1
2024,"Cafes, restaurants and takeaway food services",6113.6,12
2024,"Clothing, footwear and personal accessory retailing",2757.7,1
2024,"Clothing, footwear and personal accessory retailing",4582.3,12
2024,Department stores,1672.6,1
2024,Department stores,3216.1,12
2024,Food retailing,14188.2,1
2024,Food retailing,17040.3,12
2024,Household goods retailing,5611.9,1
2024,Household goods retailing,7645.9,12
